# FastAPI and PostgreSQL: Build Our Library API
## Week 4 · Member 6 · Maple Street Library

### First: what is FastAPI?
**FastAPI is a Python framework for building HTTP APIs.** An API lets a client ask your program to do something: find a book, register a member or borrow a copy.

FastAPI helps you:
- Match a request to a Python function.
- Read values from the URL or JSON body.
- Validate those values using Python types and Pydantic models.
- Turn Python results into HTTP responses, usually JSON.
- Generate interactive documentation at `/docs`.

FastAPI is **not** the database and does not save records automatically. **PostgreSQL** stores our records. **psycopg** is the Python driver we use to send SQL to PostgreSQL.

This lesson follows the [real Library Loans API contract](https://github.com/Chebemeze/library-loans-api/blob/api-contract-and-schema/docs/api/openapi.yaml). We create real tables and run requests against them. The earlier notebook remains archived.

**Class goal:** Define routes, explain path versus query parameters, and follow a request from HTTP through SQL to a saved record. No dictionaries stand in for the database.

## 1. What is a route?

A **route** connects an HTTP **method + URL path** to a Python function called a **handler**.

```python
@app.get("/v1/books/{isbn}")
def get_book(isbn: str):
    ...
```

Read it aloud: **“When a GET request matches this book URL, call get_book.”**

For `GET /v1/books/9780306406157`:
1. The method is GET: read something.
2. The path matches `/v1/books/{isbn}`.
3. FastAPI extracts the ISBN and passes it to the function.
4. The function queries PostgreSQL and returns a record or an error.

`GET /v1/books` and `POST /v1/books` are different routes. The GET searches books; the POST creates one. The function name itself does not choose the URL.

**Check:** Does writing a decorator automatically insert a row? **No.** We must write or call the SQL that performs the insert.

## 2. Path parameters, query parameters and bodies

| Input location | Real project request | What it does |
|---|---|---|
| **Path** | GET `/v1/books/9780306406157` | ISBN chooses one book. |
| **Query** | GET `/v1/books?search=lantern&limit=5` | Search and limit control the collection response. |
| **Body** | POST `/v1/books` with JSON | Supplies the ISBN, title and authors to create. |

A path parameter is written inside braces in the route: `{isbn}`. It is required for that route to match. Query parameters follow `?`; `&` separates them. Defaults can make query parameters optional, but they can also be required.

A request can combine locations:
```http
POST /v1/books/9780306406157/copies
Content-Type: application/json

{"barcode":"MSL-000123"}
```
ISBN comes from the **path**. Barcode comes from the **body**.

For GET `/v1/members/{member_id}/loans?open=true`, member_id is in the path; open is in the query. Do not put query strings inside the decorator.

**Ask the class:** In POST `/v1/loans`, member_id is a JSON field. Does its name make it a path parameter? **No: its declared location does.**

## 3. What is Uvicorn?

**Uvicorn is the server that listens for HTTP requests and runs the FastAPI application.** FastAPI defines the behaviour; Uvicorn makes it reachable.

```text
Browser / client
      ↓ HTTP request
Uvicorn → FastAPI route → psycopg → PostgreSQL tables
      ↑ HTTP response                 ↓ saved rows
```

For a file named `library_pg_app.py` containing `app = FastAPI()`:
```bash
python -m uvicorn library_pg_app:app --host 127.0.0.1 --port 8000 --reload
```
- `library_pg_app` names the Python module, without `.py`.
- The second `app` names its FastAPI object.
- Port 8000 is the local listening port.
- `--reload` restarts the development server after code changes; it is not a production setting.

Later we run this command. First we build the database and handlers. Notebook **TestClient** requests call the app in-process; they do not start Uvicorn. Both can use the **same PostgreSQL database**, so committed rows survive process restarts.

## 4. Our one connected library story

1. Add catalogue book **The Lantern Map**, ISBN `9780306406157`, authors `["Ada Fiction"]`.
2. Add its physical copy, barcode `MSL-000123`.
3. Register **Ada Example** and keep her returned public UUID.
4. Borrow that physical copy, creating an integer loan ID.
5. Read Ada's open loans.
6. Return the copy and check the loan, copy and possible fine together.

### Follow the real contract, not the generic old example
- Paths start with `/v1`.
- Books use ISBN strings; copies use barcodes; members use public UUIDs; loans use integers.
- Book search uses `search`; pagination defaults are limit 20 and offset 0.
- Borrow body: **copy_barcode and member_id only**. The server sets a 14-day due date.
- Return: **POST `/v1/loans/{loan_id}/return`**, not PATCH.
- Error bodies have only **code and message**.
- This contract has no book/loan DELETE or generic book PUT routes. “CRUD” in the weekly plan does not justify inventing them.

The saved `openapi.snapshot.yaml` is the teaching reference, from repository tree `b1253947270020999e78182eedc6134c2e2a08b8`. Our classroom schema is a proposal implementing this subset, not a replacement for the team's agreed Alembic migrations.

## 5. WSL setup: PostgreSQL, then Python

Run in your **Ubuntu WSL terminal**, not a Python cell:
```bash
sudo apt update
sudo apt install postgresql postgresql-contrib python3-venv
sudo service postgresql start
sudo -u postgres psql
```
Inside psql, create a dedicated learning role/database:
```sql
CREATE ROLE library_student LOGIN;
\password library_student
CREATE DATABASE library_week4 OWNER library_student;
\q
```
The password prompt does not display your password. If the role/database already exists, reuse it instead of recreating it. Check access:
```bash
psql -h 127.0.0.1 -U library_student -d library_week4
```
Inside psql, try `SELECT current_database();`, then `\q` to leave.

Extract this whole pack into `~/harbour-learning/week4-postgres`. Then:
```bash
cd ~/harbour-learning/week4-postgres
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
jupyter lab
```
Create the folder before extracting if needed. Keep the notebook and YAML together. Select this WSL Python environment if using VS Code.

**Safety:** Use only the dedicated `library_week4` database. This lesson uses persistent tables in a separate `library_lesson` schema. Do not point it at production or your teammates' shared data. No credentials are saved in this notebook.

In [1]:
import os
import getpass
from datetime import datetime, timezone, timedelta
from uuid import UUID, uuid4
from typing import Annotated
import psycopg
from psycopg.rows import dict_row

# Standard PostgreSQL environment variables; no password appears in source code.
os.environ.setdefault("PGHOST", "127.0.0.1")
os.environ.setdefault("PGPORT", "5432")
os.environ.setdefault("PGDATABASE", "library_week4")
os.environ.setdefault("PGUSER", "library_student")
if not os.environ.get("PGPASSWORD"):
    os.environ["PGPASSWORD"] = getpass.getpass("Local learning database password: ")

def get_connection():
    if os.environ.get("PGDATABASE") != "library_week4":
        raise RuntimeError("This classroom app only permits the dedicated library_week4 database.")
    return psycopg.connect(row_factory=dict_row,
        options="-c search_path=library_lesson -c timezone=UTC")

with get_connection() as connection:
    row = connection.execute("SELECT current_database() AS database, version() AS version").fetchone()
print("Connected to:", row["database"])
print("PostgreSQL:", row["version"].split(",")[0])
# getpass hides input. Never print os.environ or paste a password into a saved cell.

Connected to: library_week4
PostgreSQL: PostgreSQL 17.12 (Debian 17.12-0+deb13u1) on x86_64-pc-linux-gnu


## 6. Create books first, then physical copies

We create these tables **now**, before defining any routes.

- `books`: one catalogue record; ISBN is the key exposed by this API.
- `copies`: one physical object; barcode is its key, ISBN links it to books.

`authors TEXT[]` stores a PostgreSQL text array, matching the API's authors list. It is a classroom design choice; coordinate with the schema owner if the project normalises authors into separate tables.

A **foreign key** prevents a copy pointing to a missing book. A **CHECK** restricts copy status. These database rules still apply if someone writes SQL without using FastAPI.

`with get_connection()` opens a connection. A normal exit commits the SQL; an exception rolls it back. The connection closes afterward. These are real persistent tables, **not TEMP tables**.

`CREATE TABLE IF NOT EXISTS` lets setup run again without dropping data. It does not modify an old table to a newer design. Use migrations for real schema changes.

In [2]:
BOOKS_AND_COPIES_SQL = """
CREATE SCHEMA IF NOT EXISTS library_lesson;
CREATE TABLE IF NOT EXISTS library_lesson.books (
    isbn TEXT PRIMARY KEY CHECK (isbn ~ '^([0-9]{9}[0-9X]|[0-9]{13})$'),
    title TEXT NOT NULL CHECK (char_length(title) BETWEEN 1 AND 300),
    authors TEXT[] NOT NULL CHECK (cardinality(authors) BETWEEN 1 AND 20)
);
CREATE TABLE IF NOT EXISTS library_lesson.copies (
    barcode TEXT PRIMARY KEY CHECK (char_length(barcode) BETWEEN 1 AND 64),
    isbn TEXT NOT NULL REFERENCES library_lesson.books(isbn),
    status TEXT NOT NULL DEFAULT 'available'
        CHECK (status IN ('available', 'on_loan', 'withdrawn'))
);
"""
with get_connection() as connection:
    connection.execute(BOOKS_AND_COPIES_SQL)
    tables = connection.execute("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'library_lesson' ORDER BY table_name
    """).fetchall()
print("Tables currently created:", [row["table_name"] for row in tables])

Tables currently created: ['books', 'copies']


## 7. Add members, loans and fines

- `members.id`: a UUID generated when registering; public identity in this API.
- `loans.id`: a generated integer; links a member and a copy.
- `borrowed_at`, `due_at`, `returned_at`: UTC instants. NULL returned_at means open.
- `fines`: integer kobo; paid_kobo cannot exceed amount_kobo.

**Important database protection:** A partial unique index allows only one loan with `returned_at IS NULL` for any copy barcode. That is a backstop against lending one physical copy twice concurrently.

The **three-open-loan limit** is not an ordinary per-row CHECK. We enforce it in the borrow transaction while locking that member's row. Each borrowing path must use the same protocol; a database constraint alone does not enforce all business rules.

The optional reset below deletes only this lesson's five tables' data. Leave it False for your first run. If replaying from the beginning, change it to True deliberately, run once, then change it back. It does not drop tables or alter your production schema.

In [3]:
MEMBERS_LOANS_FINES_SQL = """
CREATE TABLE IF NOT EXISTS library_lesson.members (
    id UUID PRIMARY KEY,
    name TEXT NOT NULL CHECK (char_length(name) BETWEEN 1 AND 200),
    phone TEXT CHECK (char_length(phone) <= 20),
    status TEXT NOT NULL DEFAULT 'active' CHECK (status IN ('active', 'suspended'))
);
CREATE TABLE IF NOT EXISTS library_lesson.loans (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    member_id UUID NOT NULL REFERENCES library_lesson.members(id),
    copy_barcode TEXT NOT NULL REFERENCES library_lesson.copies(barcode),
    borrowed_at TIMESTAMPTZ NOT NULL,
    due_at TIMESTAMPTZ NOT NULL CHECK (due_at >= borrowed_at),
    returned_at TIMESTAMPTZ CHECK (returned_at >= borrowed_at)
);
CREATE UNIQUE INDEX IF NOT EXISTS one_open_loan_per_copy
    ON library_lesson.loans(copy_barcode) WHERE returned_at IS NULL;
CREATE TABLE IF NOT EXISTS library_lesson.fines (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    loan_id BIGINT NOT NULL UNIQUE REFERENCES library_lesson.loans(id),
    amount_kobo INTEGER NOT NULL CHECK (amount_kobo BETWEEN 1 AND 2000),
    paid_kobo INTEGER NOT NULL DEFAULT 0 CHECK (paid_kobo >= 0 AND paid_kobo <= amount_kobo)
);
"""
RESET_LESSON_DATA = False  # Deliberate opt-in; erases this lesson's records only.
with get_connection() as connection:
    connection.execute(MEMBERS_LOANS_FINES_SQL)
    if RESET_LESSON_DATA:
        connection.execute("""TRUNCATE library_lesson.fines, library_lesson.loans,
            library_lesson.copies, library_lesson.members, library_lesson.books RESTART IDENTITY""")
    names = connection.execute("""SELECT table_name FROM information_schema.tables
        WHERE table_schema='library_lesson' ORDER BY table_name""").fetchall()
print("Five tables:", [row["table_name"] for row in names])

Five tables: ['books', 'copies', 'fines', 'loans', 'members']


## 8. How psycopg runs SQL safely

```python
with get_connection() as connection:
    row = connection.execute(
        "SELECT * FROM books WHERE isbn = %s", (isbn,)
    ).fetchone()
```

- `%s` is a **value placeholder**. psycopg binds the separate value safely.
- `(isbn,)` is a one-item tuple; the comma matters.
- `fetchone()` returns a row or None. `fetchall()` returns a list of rows.
- `dict_row` makes fields accessible as `row["title"]` rather than numeric positions.
- `RETURNING` retrieves values from an INSERT or UPDATE, such as the new loan ID.

**Never** form SQL by inserting user input with an f-string or string concatenation. Placeholders are for values, not table names; this lesson fixes all table names.

Everything within one connection block is one transaction by default. On success it commits; if a raised error leaves the block, it rolls back. **Do not return success before the transaction has committed.**

The connection's search_path is fixed to `library_lesson`, so the unqualified table names below refer to our classroom tables. PostgreSQL's pg_catalog remains implicitly available.

In [4]:
from fastapi import FastAPI, Path, Query, Response, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field

ISBN_PATTERN = r"^(?:\d{9}[\dX]|\d{13})$"
IsbnPath = Annotated[str, Path(pattern=ISBN_PATTERN)]
BarcodePath = Annotated[str, Path(min_length=1, max_length=64)]
LoanIdPath = Annotated[int, Path(ge=1)]
Author = Annotated[str, Field(min_length=1, max_length=200)]

class BookCreate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    isbn: str = Field(pattern=ISBN_PATTERN)
    title: str = Field(min_length=1, max_length=300)
    authors: list[Author] = Field(min_length=1, max_length=20)

class CopyCreate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    barcode: str = Field(min_length=1, max_length=64)

class MemberCreate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str = Field(min_length=1, max_length=200)
    phone: str | None = Field(default=None, max_length=20)

class LoanCreate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    copy_barcode: str = Field(min_length=1, max_length=64)
    member_id: UUID

app = FastAPI(title="Maple Street — PostgreSQL routing classroom", version="3.0")

class LibraryProblem(Exception):
    def __init__(self, status_code, code, message):
        self.status_code, self.code, self.message = status_code, code, message

@app.exception_handler(LibraryProblem)
async def business_error(request: Request, error: LibraryProblem):
    return JSONResponse(status_code=error.status_code,
        content={"code": error.code, "message": error.message})

@app.exception_handler(RequestValidationError)
async def validation_error(request: Request, error: RequestValidationError):
    return JSONResponse(status_code=422,
        content={"code": "INVALID_INPUT", "message": "Check the request fields and parameter values."})

def utc_now():
    return datetime.now(timezone.utc)
clock = utc_now
print("Request models and the real contract's code/message error format are ready.")

Request models and the real contract's code/message error format are ready.


In [5]:
@app.post("/v1/books", status_code=201, operation_id="createBook")
def create_book(payload: BookCreate, response: Response):
    try:
        with get_connection() as connection:
            connection.execute("INSERT INTO books (isbn, title, authors) VALUES (%s, %s, %s)",
                               (payload.isbn, payload.title, payload.authors))
    except psycopg.errors.UniqueViolation:
        raise LibraryProblem(409, "DUPLICATE_ISBN", "A book with this ISBN already exists.")
    response.headers["Location"] = f"/v1/books/{payload.isbn}"
    return {"isbn": payload.isbn}

@app.post("/v1/books/{isbn}/copies", status_code=201, operation_id="addCopy")
def add_copy(isbn: IsbnPath, payload: CopyCreate, response: Response):
    try:
        with get_connection() as connection:
            if not connection.execute("SELECT isbn FROM books WHERE isbn=%s", (isbn,)).fetchone():
                raise LibraryProblem(404, "BOOK_NOT_FOUND", "No book with that ISBN.")
            connection.execute("INSERT INTO copies (barcode, isbn) VALUES (%s, %s)", (payload.barcode, isbn))
    except psycopg.errors.UniqueViolation:
        raise LibraryProblem(409, "DUPLICATE_BARCODE", "A copy with this barcode already exists.")
    response.headers["Location"] = f"/v1/copies/{payload.barcode}"
    return {"barcode": payload.barcode}

@app.post("/v1/members", status_code=201, operation_id="createMember")
def create_member(payload: MemberCreate, response: Response):
    member_id = uuid4()
    with get_connection() as connection:
        connection.execute("INSERT INTO members (id, name, phone) VALUES (%s, %s, %s)",
                           (member_id, payload.name, payload.phone))
    response.headers["Location"] = f"/v1/members/{member_id}"
    return {"id": member_id}

In [6]:
from fastapi.testclient import TestClient
client = TestClient(app)
ISBN, BARCODE = "9780306406157", "MSL-000123"

book_response = client.post("/v1/books", json={"isbn": ISBN,
    "title": "The Lantern Map", "authors": ["Ada Fiction"]})
assert book_response.status_code == 201, book_response.text
copy_response = client.post(f"/v1/books/{ISBN}/copies", json={"barcode": BARCODE})
assert copy_response.status_code == 201
member_response = client.post("/v1/members", json={"name": "Ada Example", "phone": "08001234567"})
assert member_response.status_code == 201
MEMBER_ID = member_response.json()["id"]
assert book_response.json() == {"isbn": ISBN}
assert book_response.headers["location"] == f"/v1/books/{ISBN}"

# NEW connection: these rows really committed to PostgreSQL.
with get_connection() as connection:
    saved_book = connection.execute("SELECT * FROM books WHERE isbn=%s", (ISBN,)).fetchone()
    saved_copy = connection.execute("SELECT * FROM copies WHERE barcode=%s", (BARCODE,)).fetchone()
    saved_member = connection.execute("SELECT * FROM members WHERE id=%s", (UUID(MEMBER_ID),)).fetchone()
assert saved_book["authors"] == ["Ada Fiction"]
assert saved_copy["status"] == "available"
assert saved_member["phone"] == "08001234567"
print("Saved book:", saved_book)
print("Saved copy:", saved_copy)
print("Ada's saved public UUID:", MEMBER_ID)
# If ISBN already exists when replaying, deliberately reset the classroom data first.

Saved book: {'isbn': '9780306406157', 'title': 'The Lantern Map', 'authors': ['Ada Fiction']}
Saved copy: {'barcode': 'MSL-000123', 'isbn': '9780306406157', 'status': 'available'}
Ada's saved public UUID: 4e03a081-5b9a-4149-9881-d1deeba4b850


## 9. Read a book; search a collection

The next routes perform SELECTs against the tables you just created.

**One resource:** GET `/v1/books/{isbn}` validates a path string and returns the book plus its physical copies. A valid-shaped but missing ISBN is 404; an invalid shape is 422.

**Collection:** GET `/v1/books?search=lantern&limit=5&offset=0` uses query values.
- Search matches title **or any author**, case-insensitively.
- Without search, list all books in pages.
- Limit defaults to 20, range 1–100. Offset defaults to 0, minimum 0.
- Return `{"items": [...], "limit": 5, "offset": 0}`, not a bare array.

SQL does the filtering/pagination instead of loading every book into Python. Our lesson chooses ISBN ordering for repeatable pages; the contract does not specify order.

The search term is passed as a bound value. We escape `%` and `_` so user text is treated as a literal substring rather than SQL wildcard instructions. Parameter binding prevents SQL injection; wildcard escaping controls search meaning—different jobs.

In [7]:
@app.get("/v1/books", operation_id="searchBooks")
def search_books(
    search: Annotated[str | None, Query(min_length=1, max_length=200)] = None,
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
    offset: Annotated[int, Query(ge=0)] = 0,
):
    with get_connection() as connection:
        if search is None:
            rows = connection.execute("SELECT * FROM books ORDER BY isbn LIMIT %s OFFSET %s",
                                      (limit, offset)).fetchall()
        else:
            literal = search.replace("!", "!!").replace("%", "!%").replace("_", "!_")
            pattern = "%" + literal + "%"
            rows = connection.execute("""
                SELECT * FROM books WHERE title ILIKE %s ESCAPE '!'
                OR EXISTS (SELECT 1 FROM unnest(authors) AS a(name) WHERE a.name ILIKE %s ESCAPE '!')
                ORDER BY isbn LIMIT %s OFFSET %s
            """, (pattern, pattern, limit, offset)).fetchall()
    return {"items": rows, "limit": limit, "offset": offset}

@app.get("/v1/books/{isbn}", operation_id="getBook")
def get_book(isbn: IsbnPath):
    with get_connection() as connection:
        book = connection.execute("SELECT * FROM books WHERE isbn=%s", (isbn,)).fetchone()
        if book is None:
            raise LibraryProblem(404, "BOOK_NOT_FOUND", "No book with that ISBN.")
        book_copies = connection.execute("SELECT * FROM copies WHERE isbn=%s ORDER BY barcode", (isbn,)).fetchall()
    return {**book, "copies": book_copies}

@app.get("/v1/copies/{barcode}", operation_id="getCopy")
def get_copy(barcode: BarcodePath):
    with get_connection() as connection:
        row = connection.execute("SELECT * FROM copies WHERE barcode=%s", (barcode,)).fetchone()
    if row is None:
        raise LibraryProblem(404, "COPY_NOT_FOUND", "No copy with that barcode.")
    return row

@app.get("/v1/members/{member_id}", operation_id="getMember")
def get_member(member_id: UUID):
    with get_connection() as connection:
        row = connection.execute("SELECT * FROM members WHERE id=%s", (member_id,)).fetchone()
    if row is None:
        raise LibraryProblem(404, "MEMBER_NOT_FOUND", "No member with that ID.")
    return row

In [8]:
book_read = client.get(f"/v1/books/{ISBN}")
assert book_read.status_code == 200
assert book_read.json()["copies"][0]["barcode"] == BARCODE
search_response = client.get("/v1/books", params={"search": "LANTERN", "limit": 5})
assert search_response.status_code == 200
assert search_response.json()["items"][0]["isbn"] == ISBN
assert search_response.json()["limit"] == 5
assert client.get("/v1/books", params={"search": "ada fiction"}).json()["items"]
assert client.get("/v1/books", params={"search": "nothing matches"}).json()["items"] == []
assert client.get("/v1/books", params={"search": "' OR 1=1 --"}).json()["items"] == []
assert client.get("/v1/books?limit=0").status_code == 422
assert client.get("/v1/books/abc").status_code == 422
assert client.get("/v1/books/9781234567897").status_code == 404
print("Read one:", book_read.json())
print("Search page:", search_response.json())
print("PASS: database reads, query validation and safely bound search input.")

Read one: {'isbn': '9780306406157', 'title': 'The Lantern Map', 'authors': ['Ada Fiction'], 'copies': [{'barcode': 'MSL-000123', 'isbn': '9780306406157', 'status': 'available'}]}
Search page: {'items': [{'isbn': '9780306406157', 'title': 'The Lantern Map', 'authors': ['Ada Fiction']}], 'limit': 5, 'offset': 0}
PASS: database reads, query validation and safely bound search input.


## 10. Borrow: one HTTP request, one database transaction

POST `/v1/loans` body:
```json
{"copy_barcode":"MSL-000123","member_id":"<Ada's public UUID>"}
```
The client does **not** choose due_at. The server records borrowing time and adds 14 days.

### Read the next handler as numbered steps
1. Lock the member row with `FOR UPDATE`.
2. Lock the copy row. Consistently lock member before copy.
3. Reject a missing member/copy with 404.
4. Reject an unavailable copy with 409 NO_COPY_AVAILABLE.
5. Reject a suspended member or unpaid fine with 409 MEMBER_BLOCKED.
6. Count open loans; reject three or more with 409 LOAN_LIMIT.
7. INSERT the loan and UPDATE the copy to on_loan in the **same transaction**.
8. Commit; return 201, identifier-only body and Location.

**Why locks?** Two requests may both ask for the last copy. A row lock makes a competing request wait; it then rechecks the committed state. Locking a member also serialises that member's loan-limit checks. The partial unique index is an additional backstop.

These rules are included so the classroom story is real. In the project, Member 1 should own the service and Member 8 the repository/session integration. Do not keep production business logic in oversized router functions.

In [9]:
@app.post("/v1/loans", status_code=201, operation_id="borrowCopy")
def borrow_copy(payload: LoanCreate, response: Response):
    with get_connection() as connection:
        member = connection.execute("SELECT * FROM members WHERE id=%s FOR UPDATE", (payload.member_id,)).fetchone()
        if member is None:
            raise LibraryProblem(404, "MEMBER_NOT_FOUND", "No member with that ID.")
        copy = connection.execute("SELECT * FROM copies WHERE barcode=%s FOR UPDATE", (payload.copy_barcode,)).fetchone()
        if copy is None:
            raise LibraryProblem(404, "COPY_NOT_FOUND", "No copy with that barcode.")
        if copy["status"] != "available":
            raise LibraryProblem(409, "NO_COPY_AVAILABLE", "This copy is not available.")
        unpaid = connection.execute("""SELECT 1 FROM fines f JOIN loans l ON l.id=f.loan_id
            WHERE l.member_id=%s AND f.paid_kobo < f.amount_kobo LIMIT 1""", (payload.member_id,)).fetchone()
        if member["status"] == "suspended" or unpaid is not None:
            raise LibraryProblem(409, "MEMBER_BLOCKED", "Member is suspended or has unpaid fines.")
        count = connection.execute("SELECT count(*) AS n FROM loans WHERE member_id=%s AND returned_at IS NULL",
                                   (payload.member_id,)).fetchone()["n"]
        if count >= 3:
            raise LibraryProblem(409, "LOAN_LIMIT", "Member already has 3 open loans.")
        borrowed_at = clock()
        loan = connection.execute("""INSERT INTO loans (member_id, copy_barcode, borrowed_at, due_at)
            VALUES (%s, %s, %s, %s) RETURNING id""",
            (payload.member_id, payload.copy_barcode, borrowed_at, borrowed_at + timedelta(days=14))).fetchone()
        connection.execute("UPDATE copies SET status='on_loan' WHERE barcode=%s", (payload.copy_barcode,))
    response.headers["Location"] = f"/v1/loans/{loan['id']}"
    return {"id": loan["id"]}

@app.get("/v1/loans/{loan_id}", operation_id="getLoan")
def get_loan(loan_id: LoanIdPath):
    with get_connection() as connection:
        row = connection.execute("SELECT * FROM loans WHERE id=%s", (loan_id,)).fetchone()
    if row is None:
        raise LibraryProblem(404, "LOAN_NOT_FOUND", "No loan with that ID.")
    return row

@app.get("/v1/members/{member_id}/loans", operation_id="listMemberLoans")
def list_member_loans(member_id: UUID,
    open_only: Annotated[bool | None, Query(alias="open")] = None,
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
    offset: Annotated[int, Query(ge=0)] = 0):
    with get_connection() as connection:
        if not connection.execute("SELECT id FROM members WHERE id=%s", (member_id,)).fetchone():
            raise LibraryProblem(404, "MEMBER_NOT_FOUND", "No member with that ID.")
        rows = connection.execute("""SELECT * FROM loans WHERE member_id=%s
            AND (%s = FALSE OR returned_at IS NULL) ORDER BY id LIMIT %s OFFSET %s""",
            (member_id, open_only is True, limit, offset)).fetchall()
    return {"items": rows, "limit": limit, "offset": offset}

In [10]:
# Freeze the SERVER clock only for a repeatable teaching test.
BORROW_TIME = datetime(2027, 1, 5, 9, tzinfo=timezone.utc)
clock = lambda: BORROW_TIME
borrow_body = {"copy_barcode": BARCODE, "member_id": MEMBER_ID}
borrow_response = client.post("/v1/loans", json=borrow_body)
assert borrow_response.status_code == 201, borrow_response.text
LOAN_ID = borrow_response.json()["id"]
assert borrow_response.headers["location"] == f"/v1/loans/{LOAN_ID}"
loan_read = client.get(borrow_response.headers["location"])
assert loan_read.status_code == 200
assert loan_read.json()["returned_at"] is None
assert client.get(f"/v1/copies/{BARCODE}").json()["status"] == "on_loan"
with get_connection() as connection:
    saved_loan = connection.execute("SELECT * FROM loans WHERE id=%s", (LOAN_ID,)).fetchone()
assert saved_loan["due_at"] == BORROW_TIME + timedelta(days=14)
repeat = client.post("/v1/loans", json=borrow_body)
assert repeat.status_code == 409 and repeat.json()["code"] == "NO_COPY_AVAILABLE"
open_page = client.get(f"/v1/members/{MEMBER_ID}/loans", params={"open": "true", "limit": 5})
assert [row["id"] for row in open_page.json()["items"]] == [LOAN_ID]
print("Saved loan:", saved_loan)
print("Repeated borrow:", repeat.json())
print("Ada's open-loan page:", open_page.json())
# open is the external query name; open_only is the Python variable.
# Omitted or false means no open-only filter in this lesson; not 'only returned'.

Saved loan: {'id': 1, 'member_id': UUID('4e03a081-5b9a-4149-9881-d1deeba4b850'), 'copy_barcode': 'MSL-000123', 'borrowed_at': datetime.datetime(2027, 1, 5, 9, 0, tzinfo=datetime.timezone.utc), 'due_at': datetime.datetime(2027, 1, 19, 9, 0, tzinfo=datetime.timezone.utc), 'returned_at': None}
Repeated borrow: {'code': 'NO_COPY_AVAILABLE', 'message': 'This copy is not available.'}
Ada's open-loan page: {'items': [{'id': 1, 'member_id': '4e03a081-5b9a-4149-9881-d1deeba4b850', 'copy_barcode': 'MSL-000123', 'borrowed_at': '2027-01-05T09:00:00+00:00', 'due_at': '2027-01-19T09:00:00+00:00', 'returned_at': None}], 'limit': 5, 'offset': 0}


## 11. Return: save three related changes together

Contract route: **POST `/v1/loans/{loan_id}/return`**, no body.

The transaction must set returned_at, free the physical copy, and create any fine. Either all three changes commit or none do.

The handler first reads the loan's immutable references, then locks **member → copy → loan**, consistent with borrowing's lock order. It rechecks returned_at after acquiring the locks. This prevents two returns both raising fines for the same loan.

Fine rule: **50 kobo per late day, capped at 2,000 kobo**. On-time return produces zero. The contract leaves partial-day rounding unspecified; this classroom version counts complete elapsed 24-hour days. Confirm that policy before production.

Success is **200 with ReturnResult**, not 204:
```json
{"loan": {"id": 1, "...": "closed loan fields"}, "fine_kobo": 0}
```
The `...` above is explanatory, not an actual JSON field.

An already returned loan produces 409 ALREADY_RETURNED. Returning twice is not “just set the same status again” in this contract.

In [11]:
@app.post("/v1/loans/{loan_id}/return", operation_id="returnLoan")
def return_loan(loan_id: LoanIdPath):
    with get_connection() as connection:
        reference = connection.execute("SELECT member_id, copy_barcode FROM loans WHERE id=%s", (loan_id,)).fetchone()
        if reference is None:
            raise LibraryProblem(404, "LOAN_NOT_FOUND", "No loan with that ID.")
        connection.execute("SELECT id FROM members WHERE id=%s FOR UPDATE", (reference["member_id"],)).fetchone()
        connection.execute("SELECT barcode FROM copies WHERE barcode=%s FOR UPDATE", (reference["copy_barcode"],)).fetchone()
        loan = connection.execute("SELECT * FROM loans WHERE id=%s FOR UPDATE", (loan_id,)).fetchone()
        if loan["returned_at"] is not None:
            raise LibraryProblem(409, "ALREADY_RETURNED", "This loan was already returned.")
        returned_at = clock()
        late_days = max(0, (returned_at - loan["due_at"]).days)
        fine_kobo = min(late_days * 50, 2000)
        closed = connection.execute("UPDATE loans SET returned_at=%s WHERE id=%s RETURNING *",
                                    (returned_at, loan_id)).fetchone()
        connection.execute("UPDATE copies SET status='available' WHERE barcode=%s", (loan["copy_barcode"],))
        if fine_kobo > 0:
            connection.execute("INSERT INTO fines (loan_id, amount_kobo) VALUES (%s, %s)", (loan_id, fine_kobo))
    return {"loan": closed, "fine_kobo": fine_kobo}

In [12]:
clock = lambda: BORROW_TIME + timedelta(days=7)
return_response = client.post(f"/v1/loans/{LOAN_ID}/return")
assert return_response.status_code == 200
assert return_response.json()["fine_kobo"] == 0
assert client.get(f"/v1/copies/{BARCODE}").json()["status"] == "available"
assert client.get(f"/v1/members/{MEMBER_ID}/loans?open=true").json()["items"] == []
assert client.post(f"/v1/loans/{LOAN_ID}/return").json()["code"] == "ALREADY_RETURNED"

# Borrow again; return exactly two days late, then inspect the real fine row.
clock = lambda: BORROW_TIME + timedelta(days=8)
late_borrow = client.post("/v1/loans", json=borrow_body)
assert late_borrow.status_code == 201
LATE_LOAN_ID = late_borrow.json()["id"]
clock = lambda: BORROW_TIME + timedelta(days=24)
late_return = client.post(f"/v1/loans/{LATE_LOAN_ID}/return")
assert late_return.status_code == 200 and late_return.json()["fine_kobo"] == 100
with get_connection() as connection:
    saved_fine = connection.execute("SELECT * FROM fines WHERE loan_id=%s", (LATE_LOAN_ID,)).fetchone()
assert saved_fine["amount_kobo"] == 100 and saved_fine["paid_kobo"] == 0
blocked = client.post("/v1/loans", json=borrow_body)
assert blocked.status_code == 409 and blocked.json()["code"] == "MEMBER_BLOCKED"
print("On-time return:", return_response.json())
print("Late-return fine saved in PostgreSQL:", saved_fine)
print("New borrowing blocked:", blocked.json())
clock = utc_now  # Live app always starts with the real clock, not these test overrides.

On-time return: {'loan': {'id': 1, 'member_id': '4e03a081-5b9a-4149-9881-d1deeba4b850', 'copy_barcode': 'MSL-000123', 'borrowed_at': '2027-01-05T09:00:00+00:00', 'due_at': '2027-01-19T09:00:00+00:00', 'returned_at': '2027-01-12T09:00:00+00:00'}, 'fine_kobo': 0}
Late-return fine saved in PostgreSQL: {'id': 1, 'loan_id': 2, 'amount_kobo': 100, 'paid_kobo': 0}
New borrowing blocked: {'code': 'MEMBER_BLOCKED', 'message': 'Member is suspended or has unpaid fines.'}


## 12. Use Uvicorn and see the SAME saved data

After running the notebook, open a **second WSL terminal** in the extracted folder:
```bash
source .venv/bin/activate
export PGHOST=127.0.0.1 PGPORT=5432 PGDATABASE=library_week4 PGUSER=library_student
read -rsp "Local database password: " PGPASSWORD
printf '\n'
export PGPASSWORD
python -m uvicorn library_pg_app:app --host 127.0.0.1 --port 8000 --reload
```
The prompt hides the password; do not echo the environment or save credentials in Git. Run `unset PGPASSWORD` after stopping the server. Environment credentials are for this local lesson, not a full production secret-management policy.

Open `http://localhost:8000/docs`. GET `/v1/books` should show the notebook's committed book because both processes use the same database/schema. Read the returned member/loan IDs from the notebook output; do not assume the ID values.

**Persistence test:** Stop Uvicorn with Ctrl+C, start it again, and GET the book. It should still be there. Restarting the server does not delete PostgreSQL rows. Conversely, deleting/resetting the classroom rows affects both processes.

For a live borrow demo after the notebook, register a **new member** and add a new barcode: Ada now has an unpaid fine and is correctly blocked. Do not present that expected block as a broken route.

`/docs` is generated from the implemented subset, not the complete official YAML. This app deliberately omits `/health/ready` until its explicit database-probe implementation is added; do not claim a readiness route you have not written.

## 13. Three exercises for the class

### A — Explain inputs, then make the requests
1. Search by an author's name with limit 1.
2. Read one book by ISBN.
3. Add another copy: explain why ISBN is in the path and barcode in the body.
4. Read a member's open loans: identify the UUID path and boolean query.

### B — Prove failure rolls back
Using a fresh member and available copy, temporarily add `raise RuntimeError("deliberate failure")` after the borrow INSERT but before the copy UPDATE in a **throwaway local version**.
- Predict the result: no loan row should commit and the copy should remain available.
- TestClient raises server exceptions by default; catch the deliberate error, or use `TestClient(app, raise_server_exceptions=False)` to inspect HTTP 500.
- Query PostgreSQL to verify both facts. Remove the injected failure and rerun.
- Do not catch and suppress the exception inside the transaction block; it must escape so rollback occurs.

### C — Implement the actual member PATCH
Consult `MemberUpdate` in the YAML. Add PATCH `/v1/members/{member_id}` with parameterised UPDATE SQL.
- At least one field; only name and phone.
- Omitted phone stays unchanged; explicit null clears it.
- Null name is invalid. Return the full member; unknown UUID gives 404 MEMBER_NOT_FOUND.
- Tests: name-only update, clear phone, empty body, null name, missing UUID.

**Before code:** Write the expected request and response. **After code:** inspect the saved row. Do not merely check HTTP 200.

In [13]:
# Check response samples against the actual project's saved JSON Schemas.
import yaml
from pathlib import Path as FilePath
from jsonschema import Draft202012Validator, FormatChecker
contract = yaml.safe_load(FilePath("openapi.snapshot.yaml").read_text())
def check_schema(response, schema_name):
    schema = {"$ref": f"#/components/schemas/{schema_name}", "components": contract["components"]}
    Draft202012Validator(schema, format_checker=FormatChecker()).validate(response.json())
for response, name in [(book_response, "CreatedBook"), (copy_response, "CreatedCopy"),
    (member_response, "CreatedMember"), (book_read, "Book"), (search_response, "BookPage"),
    (borrow_response, "CreatedLoan"), (loan_read, "Loan"), (open_page, "LoanPage"),
    (return_response, "ReturnResult"), (late_return, "ReturnResult"), (blocked, "Error")]:
    check_schema(response, name)

# Two different members compete for ONE available copy, using actual PostgreSQL locks.
from concurrent.futures import ThreadPoolExecutor
race_members = [client.post("/v1/members", json={"name": name}).json()["id"]
                for name in ["Boma Example", "Chidi Example"]]
assert client.post(f"/v1/books/{ISBN}/copies", json={"barcode": "MSL-RACE-001"}).status_code == 201

def compete(member_id):
    with TestClient(app) as competing_client:
        return competing_client.post("/v1/loans", json={"member_id": member_id,
                                                       "copy_barcode": "MSL-RACE-001"})
with ThreadPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(compete, race_members))
assert sorted(response.status_code for response in results) == [201, 409]
assert next(response for response in results if response.status_code == 409).json()["code"] == "NO_COPY_AVAILABLE"
with get_connection() as connection:
    active = connection.execute("SELECT count(*) AS n FROM loans WHERE copy_barcode=%s AND returned_at IS NULL",
                                ("MSL-RACE-001",)).fetchone()["n"]
assert active == 1
print("PASS: selected responses match contract schemas.")
print("PASS: two real database requests for one copy -> one success, one conflict, one open loan.")
client.close()
# This small concurrent smoke test is not an exhaustive concurrency or load test.

PASS: selected responses match contract schemas.
PASS: two real database requests for one copy -> one success, one conflict, one open loan.


## 14. What to explain before your Friday merge

Use this script: **“This request matches this route. These inputs come from the path/query/body. psycopg binds them to this SQL. PostgreSQL enforces these rules. The transaction commits these changes. The response matches this contract.”**

Ask your team:
1. What is FastAPI, and how is it different from Uvicorn? *(Framework versus HTTP server.)*
2. Why is search a query but ISBN a path value here? *(Selection/options versus one resource.)*
3. What does psycopg do? *(Connect Python to PostgreSQL and execute SQL.)*
4. Why use `%s` plus a tuple instead of SQL f-strings? *(Safely bind data values.)*
5. When does the connection block commit or roll back? *(Normal exit versus an escaping exception.)*
6. Why do records survive restarting the API now? *(Committed rows live in PostgreSQL.)*
7. Why lock the member and copy while borrowing? *(Serialise limit/availability checks.)*
8. Why not add DELETE just because the plan says CRUD? *(It is absent from the current contract.)*

**Implemented subset:** create/search/read books, add/read copies, register/read members, borrow/read loans, member-loan query, return with saved fines. There are no database stand-ins.

**Not complete:** Member PATCH, suspend/reactivate, fine listing/payments, readiness probe, full response models/generated OpenAPI documentation, database connection pooling, production observability/error mapping, and the rest of the API. Request validation follows the basic contract, not ISBN checksum rules. Partial late-day rounding and stable pagination order need team agreement. Per-author text limits are validated by the API, not comprehensively by the array column constraint.

This is synchronous `def` route code with synchronous psycopg; do not place blocking calls directly inside an `async def` handler. Per-request connections are simple for teaching; production connection/pool management needs review.

**Team handoff:** Member 1 separates services, Member 7 reviews validation, Member 8 integrates the approved repository/session approach, Member 2 expands database tests. The team's plan uses SQLAlchemy in the integrated app; this direct-psycopg lesson deliberately exposes the SQL first. Classroom CREATE TABLE statements are not an alternative migration history: transfer approved designs through Alembic with the schema owner.

**Security:** The contract has no login. Anyone who reaches it can change loans. Keep it local/private. Fictional data only. Do not commit passwords or publish this classroom app.

**Honest evidence:** Run all cells against your local PostgreSQL. The last code cell checks selected responses and one concurrent-borrow scenario; it does not prove every production behaviour. Record your own environment and tests, explain code live, and disclose AI assistance.

Sources: [Library contract](https://github.com/Chebemeze/library-loans-api/blob/api-contract-and-schema/docs/api/openapi.yaml), [FastAPI](https://fastapi.tiangolo.com/), [psycopg transactions](https://www.psycopg.org/psycopg3/docs/basic/transactions.html), [Uvicorn](https://www.uvicorn.org/settings/).